# LLM Pre-Training Data Engineering
## A PhD-Level Notebook

**Topics covered**
1. Training data curation pipelines (Common Crawl / The Pile / RedPajama style)
2. Data deduplication: exact hash, MinHash LSH, fuzzy clustering
3. Data quality filtering: perplexity filtering, classifier-based filtering
4. Curriculum learning and domain mixing strategies
5. Training stability: loss spikes, gradient clipping, LR warm-up

**Prerequisites:** transformers, backpropagation, basic ML.  
No prior knowledge of the specific techniques is assumed.

Run the cells top-to-bottom in Google Colab (Runtime -> Run all).

# 1) Setup

Standard imports and reproducibility seed.  
We use only PyTorch + Python stdlib + matplotlib/numpy for all core algorithms.

In [ ]:
# ── standard library ──────────────────────────────────────────────────────
import hashlib, math, random, re, string, time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set, Tuple

# ── third-party ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# ── reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# 2) Training Data Curation Pipelines

## Intuition
Large language models are trained on massive, heterogeneous web corpora
(Common Crawl, C4, The Pile, RedPajama).  Raw web text contains boilerplate,
spam, non-natural language, and near-duplicate content.  A curation pipeline
applies a cascade of cheap heuristic filters to discard low-quality documents
before expensive model-based filtering.

## Real-world examples
- **C4** (T5): language detection, dedup, sentence-ending heuristics
- **Gopher** (DeepMind): word-length ratio, symbol ratio, n-gram repetition
- **RedPajama-v2**: 20+ quality signals stored as metadata per document

## Pipeline structure
```
Raw corpus
  --> length filter
  --> symbol ratio filter
  --> mean word-length filter
  --> repetition filter  (Gopher top n-gram fraction)
  --> unique-word ratio filter
  --> [dedup stage -- Section 3]
  --> [quality scoring -- Section 4]
  --> Curated corpus
```

**Sample input:** `{"text": "Buy cheap meds now!!! click here click here click here"}`  
**Sample output:** document rejected at *repetition* filter.

In [ ]:
# ── synthetic corpus: three quality tiers ─────────────────────────────────
HIGH_QUALITY = [
    "The transformer architecture introduced in 'Attention Is All You Need' "
    "replaced recurrent networks with self-attention, enabling parallelisation "
    "across sequence positions and achieving state-of-the-art results on "
    "machine translation benchmarks.",
    "Backpropagation through time unrolls a recurrent network across timesteps "
    "and applies the chain rule to compute gradients with respect to parameters "
    "shared across all time steps.",
    "Scaling laws for neural language models suggest that loss decreases as a "
    "power law with compute, dataset size, and model parameters, motivating "
    "ever-larger training runs.",
    "Constitutional AI aligns language models through a two-stage process: "
    "supervised fine-tuning on human-preferred responses followed by "
    "reinforcement learning from AI feedback.",
    "The mixture-of-experts architecture conditionally activates a small subset "
    "of parameters per token, decoupling parameter count from compute cost "
    "and enabling trillion-parameter models to be trained feasibly.",
]

MEDIUM_QUALITY = [
    "machine learning is cool and neural networks can do lots of things "
    "like image recognition and text generation and many other tasks too.",
    "python is a programming language. it is popular. many people use it. "
    "you can do data science with python. scikit-learn is a library for ml.",
    "the model was trained on lots of data and it works pretty well for "
    "most tasks but sometimes it makes mistakes which is a problem.",
    "researchers at various universities and companies have published papers "
    "about deep learning and it has improved a lot in recent years.",
    "gradient descent is an optimisation algorithm used to train neural "
    "networks by minimising the loss function step by step.",
]

LOW_QUALITY = [
    "buy cheap meds online now!!! click here click here click here "
    "best prices best prices best prices!!!! 100% guaranteed",
    "lol omg this is so funny hahaha i cant believe it lol lol lol "
    "lmao lmao lmao lmao check this out check this out",
    "????????????????????????????????????????????????????????????????",
    "1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 "
    "25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40",
    "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa"
    "bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb",
]

# inject near-duplicate pair to stress-test deduplication later
NEAR_DUP_A = (
    "The attention mechanism allows each position in the encoder to attend "
    "to all positions in the previous layer of the encoder."
)
NEAR_DUP_B = (
    "The attention mechanism allows each position in the encoder to attend "
    "to all positions in the previous layer of the encoder!!"
)

RAW_CORPUS: List[Dict] = []
for i, text in enumerate(HIGH_QUALITY):
    RAW_CORPUS.append({"id": f"hq_{i}", "text": text, "tier": "high"})
for i, text in enumerate(MEDIUM_QUALITY):
    RAW_CORPUS.append({"id": f"mq_{i}", "text": text, "tier": "medium"})
for i, text in enumerate(LOW_QUALITY):
    RAW_CORPUS.append({"id": f"lq_{i}", "text": text, "tier": "low"})
RAW_CORPUS.append({"id": "dup_a", "text": NEAR_DUP_A, "tier": "high"})
RAW_CORPUS.append({"id": "dup_b", "text": NEAR_DUP_B, "tier": "high"})

print(f"Raw corpus: {len(RAW_CORPUS)} documents")

In [ ]:
# ── FilterConfig: all heuristic thresholds in one place ───────────────────
@dataclass
class FilterConfig:
    min_words: int = 10  # minimum word count
    max_words: int = 100_000  # maximum word count
    max_symbol_ratio: float = 0.10  # fraction of non-alphanumeric chars
    min_mean_word_len: float = 3.0  # mean word length in characters
    max_mean_word_len: float = 15.0
    max_dup_ngram_frac: float = 0.30  # Gopher top n-gram duplication fraction
    ngram_size: int = 5  # n for the repetition check
    min_unique_word_ratio: float = 0.20  # unique words / total words


CFG = FilterConfig()


# ── individual filter functions ───────────────────────────────────────────
def filter_length(text: str, cfg: FilterConfig) -> bool:
    """Accept if word count is within [min_words, max_words]."""
    n = len(text.split())
    return cfg.min_words <= n <= cfg.max_words


def filter_symbol_ratio(text: str, cfg: FilterConfig) -> bool:
    """Reject documents where non-alphanumeric chars exceed threshold.

    High symbol ratio indicates boilerplate, code dumps, or garbled text.
    """
    if not text:
        return False
    non_alpha = sum(1 for c in text if not c.isalnum() and not c.isspace())
    return (non_alpha / len(text)) <= cfg.max_symbol_ratio


def filter_word_length(text: str, cfg: FilterConfig) -> bool:
    """Reject if mean word length is outside a plausible natural-language range.

    Very short mean length -> single-character tokens or numeric lists.
    Very long mean length  -> encoded binary data or hash strings.
    """
    words = text.split()
    if not words:
        return False
    mean_len = sum(len(w) for w in words) / len(words)
    return cfg.min_mean_word_len <= mean_len <= cfg.max_mean_word_len


def filter_repetition(text: str, cfg: FilterConfig) -> bool:
    """Gopher-style top-n-gram duplication fraction.

    Compute the fraction of tokens covered by the single most frequent n-gram.
    A high fraction indicates the document is highly repetitive.
    Fraction = (count_of_most_frequent_ngram * n) / total_tokens
    """
    words = text.lower().split()
    n, total = cfg.ngram_size, len(words)
    if total < n:
        return True  # too short to judge, pass through
    ngrams = [tuple(words[i : i + n]) for i in range(total - n + 1)]
    most_common_count = Counter(ngrams).most_common(1)[0][1]
    dup_frac = (most_common_count * n) / total
    return dup_frac <= cfg.max_dup_ngram_frac


def filter_unique_words(text: str, cfg: FilterConfig) -> bool:
    """Reject documents with very low lexical diversity.

    unique_ratio = |unique tokens| / |total tokens|
    Low diversity signals spam or auto-generated repetitive text.
    """
    words = text.lower().split()
    if not words:
        return False
    return len(set(words)) / len(words) >= cfg.min_unique_word_ratio


# ── pipeline with short-circuit evaluation ────────────────────────────────
FILTERS = [
    ("length", filter_length),
    ("symbol_ratio", filter_symbol_ratio),
    ("word_length", filter_word_length),
    ("repetition", filter_repetition),
    ("unique_words", filter_unique_words),
]


def apply_heuristic_filters(doc: Dict, cfg: FilterConfig) -> Tuple[bool, Optional[str]]:
    """Run each filter in order; return (passed, rejection_reason).

    Short-circuits on first failure for efficiency.
    """
    text = doc["text"]
    for name, fn in FILTERS:
        if not fn(text, cfg):
            return False, name
    return True, None


# ── apply to corpus ───────────────────────────────────────────────────────
rejection_counts: Counter = Counter()
heuristic_passed: List[Dict] = []

for doc in RAW_CORPUS:
    passed, reason = apply_heuristic_filters(doc, CFG)
    if passed:
        heuristic_passed.append(doc)
    else:
        rejection_counts[reason] += 1

print(f"Before heuristic filters : {len(RAW_CORPUS)}")
print(f"After  heuristic filters : {len(heuristic_passed)}")
print(f"Rejections by filter     : {dict(rejection_counts)}")

In [ ]:
# ── visualise filter rejection breakdown ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# rejection breakdown bar chart
if rejection_counts:
    labels = list(rejection_counts.keys())
    values = [rejection_counts[l] for l in labels]
    axes[0].bar(labels, values, color="#e07b54", edgecolor="white")
    axes[0].set_title("Heuristic Filter Rejections")
    axes[0].set_ylabel("Documents rejected")
    axes[0].tick_params(axis="x", rotation=20)
else:
    axes[0].text(0.5, 0.5, "No rejections", ha="center", va="center")
    axes[0].set_title("Heuristic Filter Rejections")

# word-count distribution by tier
tier_colors = {"high": "#4CAF50", "medium": "#FFC107", "low": "#F44336"}
for doc in RAW_CORPUS:
    n = len(doc["text"].split())
    color = tier_colors.get(doc["tier"], "gray")
    axes[1].scatter(RAW_CORPUS.index(doc), n, color=color, alpha=0.7, s=60)

from matplotlib.patches import Patch

legend = [Patch(color=v, label=k) for k, v in tier_colors.items()]
axes[1].legend(handles=legend)
axes[1].set_title("Word Count by Quality Tier")
axes[1].set_xlabel("Document index")
axes[1].set_ylabel("Word count")
axes[1].axhline(
    CFG.min_words, color="gray", linestyle="--", alpha=0.5, label=f"min={CFG.min_words}"
)

plt.tight_layout()
plt.show()
print(f"Passed to deduplication stage: {len(heuristic_passed)} documents")

# 3) Data Deduplication

## Intuition
Web crawls contain massive amounts of duplicate and near-duplicate content:
mirrors, boilerplate repeated across pages, scraped forums.  Training on
duplicates wastes compute and causes models to memorise specific passages.

Three approaches at increasing fuzziness:

| Method | Complexity | Catches |
|---|---|---|
| Exact hash (SHA-256) | O(n) | Byte-identical docs |
| MinHash LSH | O(n) | Docs with Jaccard sim > threshold |
| Union-Find clustering | O(n*alpha) | Connected components of near-dups |

## Jaccard Similarity
For two documents A, B represented as sets of character shingles (k-grams):

    J(A, B) = |A ∩ B| / |A ∪ B|

MinHash approximates J(A,B) in sub-linear space using random hash functions.

## MinHash Theory
For a random permutation pi of the shingle universe, define:
    h(S) = min_{x in S} pi(x)

Then:  P[ h(A) == h(B) ] = J(A, B)

We use *num_hashes* independent hash functions to estimate J.
LSH banding: split the signature matrix into *b* bands of *r* rows each.
Two documents become candidates iff they agree in at least one entire band.

    P(candidate | J=s) = 1 - (1 - s^r)^b

**Sample input:** two documents sharing most words  
**Sample output:** identified as near-duplicate pair, one removed.

In [ ]:
# ── exact deduplication via SHA-256 ──────────────────────────────────────
def normalise_text(text: str) -> str:
    """Lower-case, collapse whitespace -- canonical form for hashing."""
    return " ".join(text.lower().split())


def exact_dedup(docs: List[Dict]) -> List[Dict]:
    """Remove byte-identical documents (after normalisation).

    O(n) time using a hash set. Returns deduplicated list preserving order.
    """
    seen: Set[str] = set()
    unique: List[Dict] = []
    for doc in docs:
        digest = hashlib.sha256(normalise_text(doc["text"]).encode()).hexdigest()
        if digest not in seen:
            seen.add(digest)
            unique.append(doc)
    return unique


exact_deduped = exact_dedup(heuristic_passed)
print(
    f"After exact dedup: {len(exact_deduped)} (removed {len(heuristic_passed)-len(exact_deduped)})"
)

In [ ]:
# ── MinHash from scratch ──────────────────────────────────────────────────

# Mersenne prime used as the hash universe; large enough to avoid collisions
MERSENNE_PRIME = (1 << 61) - 1  # 2^61 - 1


@dataclass
class LSHConfig:
    num_hashes: int = 128  # number of independent hash functions
    num_bands: int = 16  # number of LSH bands  (b)
    shingle_size: int = 5  # character n-gram size
    threshold: float = 0.65  # Jaccard similarity threshold

    @property
    def rows_per_band(self) -> int:  # r = num_hashes / num_bands
        assert self.num_hashes % self.num_bands == 0, "num_hashes must divide num_bands"
        return self.num_hashes // self.num_bands


def generate_hash_params(num_hashes: int, prime: int) -> Tuple[np.ndarray, np.ndarray]:
    """Sample (a, b) pairs for universal hash family  h(x) = (ax + b) mod p.

    Each pair defines one independent MinHash function.
    Output shapes: (num_hashes,), (num_hashes,)
    """
    rng = np.random.default_rng(SEED)
    a = rng.integers(1, prime, size=num_hashes)  # (num_hashes,)
    b = rng.integers(0, prime, size=num_hashes)  # (num_hashes,)
    return a, b


def get_char_shingles(text: str, k: int) -> Set[int]:
    """Extract character k-grams and map each to an integer ID.

    Using hash() on the shingle string gives a compact integer representation.
    """
    text = normalise_text(text)
    return {hash(text[i : i + k]) % MERSENNE_PRIME for i in range(len(text) - k + 1)}


def compute_minhash_signature(
    shingles: Set[int], a: np.ndarray, b: np.ndarray, prime: int
) -> np.ndarray:
    """Compute the MinHash signature vector for one document.

    For each hash function h_i(x) = (a_i*x + b_i) mod prime, take the minimum
    over all shingles x in the document.

    shingles: set of integer shingle IDs
    a, b    : (num_hashes,) parameter arrays
    Returns : (num_hashes,) signature vector
    """
    shingle_arr = np.array(list(shingles), dtype=np.int64)  # (num_shingles,)

    # broadcast: (num_hashes, 1) * (num_shingles,) -> (num_hashes, num_shingles)
    # then take min over shingle axis -> (num_hashes,)
    hashed = (
        a[:, None] * shingle_arr[None, :] + b[:, None]
    ) % prime  # (num_hashes, num_shingles)
    return hashed.min(axis=1)  # (num_hashes,)


def build_minhash_signatures(
    docs: List[Dict], cfg: LSHConfig
) -> Tuple[np.ndarray, List[str]]:
    """Build signature matrix for all documents.

    Returns:
        sig_matrix : (num_docs, num_hashes)
        doc_ids    : list of document ids matching row order
    """
    a, b = generate_hash_params(cfg.num_hashes, MERSENNE_PRIME)
    signatures, doc_ids = [], []
    for doc in docs:
        shingles = get_char_shingles(doc["text"], cfg.shingle_size)
        if not shingles:
            continue
        sig = compute_minhash_signature(shingles, a, b, MERSENNE_PRIME)
        signatures.append(sig)
        doc_ids.append(doc["id"])

    sig_matrix = np.stack(signatures, axis=0)  # (num_docs, num_hashes)
    return sig_matrix, doc_ids


LSH_CFG = LSHConfig(num_hashes=128, num_bands=16, threshold=0.65)
sig_matrix, doc_ids = build_minhash_signatures(exact_deduped, LSH_CFG)
print(f"Signature matrix shape: {sig_matrix.shape}  (num_docs x num_hashes)")

In [ ]:
# ── LSH Banding ──────────────────────────────────────────────────────────
def lsh_band_buckets(sig_matrix: np.ndarray, cfg: LSHConfig) -> Dict[Tuple, List[int]]:
    """Hash each band of the signature matrix into buckets.

    Documents that share a bucket in any band become candidate pairs.
    sig_matrix shape: (num_docs, num_hashes)
    """
    num_docs = sig_matrix.shape[0]
    r = cfg.rows_per_band
    buckets: Dict[Tuple, List[int]] = defaultdict(list)

    for band_idx in range(cfg.num_bands):
        col_start = band_idx * r
        col_end = col_start + r
        band_slice = sig_matrix[:, col_start:col_end]  # (num_docs, r)

        for doc_idx in range(num_docs):
            # key = (band_id, hash of band signature tuple)
            band_key = (band_idx, tuple(band_slice[doc_idx].tolist()))
            buckets[band_key].append(doc_idx)

    return buckets


def find_candidate_pairs(buckets: Dict[Tuple, List[int]]) -> Set[Tuple[int, int]]:
    """Collect all (i, j) pairs that share at least one LSH bucket."""
    candidates: Set[Tuple[int, int]] = set()
    for indices in buckets.values():
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                a, b = sorted([indices[i], indices[j]])
                candidates.add((a, b))
    return candidates


buckets = lsh_band_buckets(sig_matrix, LSH_CFG)
candidate_pairs = find_candidate_pairs(buckets)
print(f"LSH candidate pairs: {len(candidate_pairs)}")

In [ ]:
# ── Union-Find with path compression for connected-component clustering ──
class UnionFind:
    """Disjoint-set data structure with path compression and union-by-rank.

    Clusters near-duplicate candidates into connected components so that
    the entire group (not just pairwise) can be deduplicated.
    """

    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # path compression
        return self.parent[x]

    def union(self, x: int, y: int) -> None:
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1


def jaccard_from_signatures(sig_a: np.ndarray, sig_b: np.ndarray) -> float:
    """Estimate Jaccard similarity from MinHash signatures.

    J_hat = fraction of hash functions where sig_a[i] == sig_b[i]
    """
    return float(np.mean(sig_a == sig_b))


def fuzzy_dedup(
    docs: List[Dict],
    sig_matrix: np.ndarray,
    doc_ids: List[str],
    candidate_pairs: Set[Tuple[int, int]],
    cfg: LSHConfig,
) -> List[Dict]:
    """Full fuzzy deduplication pipeline.

    1. For each LSH candidate pair, compute exact MinHash Jaccard estimate.
    2. If estimated J >= threshold, mark as near-duplicate via Union-Find.
    3. Keep one representative per cluster (lowest index).
    """
    id_to_idx = {doc_id: i for i, doc_id in enumerate(doc_ids)}
    uf = UnionFind(len(docs))

    near_dup_pairs = []
    for i, j in candidate_pairs:
        sim = jaccard_from_signatures(sig_matrix[i], sig_matrix[j])
        if sim >= cfg.threshold:
            uf.union(i, j)
            near_dup_pairs.append((doc_ids[i], doc_ids[j], sim))

    # keep only the root document in each cluster
    kept_indices: Set[int] = {uf.find(i) for i in range(len(docs))}
    result = [docs[i] for i in range(len(docs)) if i in kept_indices]

    print(f"Near-duplicate pairs detected : {len(near_dup_pairs)}")
    for a_id, b_id, sim in near_dup_pairs:
        print(f"  '{a_id}' ~ '{b_id}'  estimated_J={sim:.3f}")
    return result


deduped_corpus = fuzzy_dedup(
    exact_deduped, sig_matrix, doc_ids, candidate_pairs, LSH_CFG
)
print(
    f"\nAfter fuzzy dedup: {len(deduped_corpus)} documents "
    f"(removed {len(exact_deduped)-len(deduped_corpus)})"
)

In [ ]:
# ── visualise LSH S-curve and MinHash error ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# S-curve: P(candidate | J=s) = 1 - (1 - s^r)^b
s_vals = np.linspace(0, 1, 200)
for b, r, style in [(8, 16, "--"), (16, 8, "-"), (32, 4, ":")]:
    p_cand = 1 - (1 - s_vals**r) ** b
    axes[0].plot(s_vals, p_cand, style, label=f"b={b}, r={r}")
axes[0].axvline(
    LSH_CFG.threshold,
    color="red",
    alpha=0.5,
    linestyle="--",
    label=f"threshold={LSH_CFG.threshold}",
)
axes[0].set_xlabel("Jaccard similarity (s)")
axes[0].set_ylabel("P(candidate pair)")
axes[0].set_title("LSH S-Curve: banding configurations")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# MinHash estimation error vs num_hashes
nh_vals = [8, 16, 32, 64, 128, 256, 512]
true_J = 0.7
std_err = [math.sqrt(true_J * (1 - true_J) / nh) for nh in nh_vals]
axes[1].plot(nh_vals, std_err, "o-", color="#e07b54")
axes[1].set_xscale("log", base=2)
axes[1].set_xlabel("num_hashes")
axes[1].set_ylabel("Std error of J estimate")
axes[1].set_title(f"MinHash Error Decay  (true J={true_J})")
axes[1].grid(alpha=0.3)

# pipeline funnel
stages = ["Raw", "Heuristic", "Exact dedup", "Fuzzy dedup"]
counts = [
    len(RAW_CORPUS),
    len(heuristic_passed),
    len(exact_deduped),
    len(deduped_corpus),
]
colors = ["#5c85d6", "#4CAF50", "#FFC107", "#e07b54"]
axes[2].barh(stages[::-1], counts[::-1], color=colors[::-1], edgecolor="white")
axes[2].set_xlabel("Documents remaining")
axes[2].set_title("Deduplication Pipeline Funnel")
for i, v in enumerate(counts[::-1]):
    axes[2].text(v + 0.1, i, str(v), va="center")
axes[2].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

# 4) Data Quality Filtering

## Intuition
After deduplication, documents vary enormously in quality.  Two complementary
approaches are widely used:

**Perplexity filtering** (CCNet, RedPajama)  
Train a small n-gram language model on a reference *clean* corpus (e.g.
Wikipedia).  Documents with very low perplexity are suspiciously templated;
very high perplexity are garbled or non-English.  Only the middle band is kept.

**Classifier-based filtering** (Gopher, LLaMA)  
Train a binary classifier (high-quality vs web-noise) on curated seed data,
then score every document.  LLaMA 1 used a fastText classifier trained on
Wikipedia vs CommonCrawl.

## Perplexity Formula
For document w_1, ..., w_T under n-gram LM P:

    PPL = exp( -1/T * sum_{t=1}^{T} log P(w_t | w_{t-n+1}, ..., w_{t-1}) )

**Sample input:** `"buy meds cheap click click click"`  
**Sample output:** PPL=8423 -> rejected (above max threshold).

In [ ]:
# ── N-gram language model (from scratch) ────────────────────────────────
class NGramLanguageModel:
    """Smoothed n-gram language model for perplexity scoring.

    Uses add-k (Laplace) smoothing to handle unseen n-grams.
    Special tokens <BOS> and <EOS> pad the context window.
    """

    BOS = "<BOS>"
    EOS = "<EOS>"
    UNK = "<UNK>"

    def __init__(self, n: int = 3, k: float = 0.5):
        self.n = n
        self.k = k  # smoothing constant
        self.vocab: Set[str] = set()
        self.context_counts: Counter = Counter()  # (n-1)-gram counts
        self.ngram_counts: Counter = Counter()  # n-gram counts
        self._fitted = False

    def _tokenise(self, text: str) -> List[str]:
        """Simple whitespace tokeniser, lower-cased."""
        return text.lower().split()

    def _pad(self, tokens: List[str]) -> List[str]:
        """Add (n-1) BOS tokens at start and one EOS at end."""
        return [self.BOS] * (self.n - 1) + tokens + [self.EOS]

    def fit(self, texts: List[str]) -> "NGramLanguageModel":
        """Estimate n-gram probabilities from a list of training texts."""
        for text in texts:
            tokens = self._pad(self._tokenise(text))
            self.vocab.update(tokens)
            for i in range(len(tokens) - self.n + 1):
                ng = tuple(tokens[i : i + self.n])
                ctx = ng[:-1]
                self.ngram_counts[ng] += 1
                self.context_counts[ctx] += 1
        self.vocab_size = len(self.vocab)
        self._fitted = True
        return self

    def log_prob(self, ngram: Tuple[str, ...]) -> float:
        """Add-k smoothed log-probability of the n-gram."""
        ctx = ngram[:-1]
        num = self.ngram_counts[ngram] + self.k
        den = self.context_counts[ctx] + self.k * self.vocab_size
        return math.log(num / den)

    def perplexity(self, text: str) -> float:
        """Compute perplexity of a text under this n-gram model.

        Unknown tokens are replaced with UNK (which has zero training count).
        """
        assert self._fitted, "Call .fit() first"
        tokens = self._pad(self._tokenise(text))
        log_prob_sum = 0.0
        count = 0
        for i in range(self.n - 1, len(tokens)):
            ng = tuple(tokens[i - self.n + 1 : i + 1])
            log_prob_sum += self.log_prob(ng)
            count += 1
        if count == 0:
            return float("inf")
        return math.exp(-log_prob_sum / count)


# train reference LM on high-quality documents only
ref_texts = [d["text"] for d in RAW_CORPUS if d["tier"] == "high"]
lm = NGramLanguageModel(n=3, k=0.5).fit(ref_texts)

# score all documents
for doc in deduped_corpus:
    doc["ppl"] = lm.perplexity(doc["text"])

sample_ppls = [(d["tier"], round(d["ppl"], 1)) for d in deduped_corpus]
print("Sample perplexities (tier, ppl):")
for t, p in sorted(sample_ppls, key=lambda x: x[1]):
    print(f"  tier={t:6s}  ppl={p}")

In [ ]:
# ── perplexity filter ────────────────────────────────────────────────────
PPL_MIN, PPL_MAX = 4.0, 600.0


def perplexity_filter(
    docs: List[Dict], ppl_min: float, ppl_max: float
) -> Tuple[List[Dict], List[Dict]]:
    """Accept documents whose perplexity falls in [ppl_min, ppl_max].

    Below ppl_min: too templated / low entropy.
    Above ppl_max: garbled, foreign, or non-natural language.
    """
    passed, rejected = [], []
    for doc in docs:
        if ppl_min <= doc["ppl"] <= ppl_max:
            passed.append(doc)
        else:
            rejected.append(doc)
    return passed, rejected


ppl_passed, ppl_rejected = perplexity_filter(deduped_corpus, PPL_MIN, PPL_MAX)
print(
    f"After perplexity filter: {len(ppl_passed)} passed, {len(ppl_rejected)} rejected"
)

In [ ]:
# ── TF-IDF Vectoriser from scratch ───────────────────────────────────────
class TFIDFVectorizer:
    """Bag-of-words TF-IDF with smooth IDF and L2 normalisation.

    Uses unigrams and bigrams (1+2-grams) as features.
    smooth_idf: IDF(t) = log( (1+N)/(1+df(t)) ) + 1
    """

    def __init__(self, max_features: int = 500, ngram_range=(1, 2)):
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.vocab_: Dict[str, int] = {}
        self.idf_: np.ndarray = np.array([])
        self._fitted = False

    def _get_ngrams(self, text: str) -> List[str]:
        tokens = text.lower().split()
        grams = []
        for n in range(self.ngram_range[0], self.ngram_range[1] + 1):
            grams += [" ".join(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]
        return grams

    def fit(self, texts: List[str]) -> "TFIDFVectorizer":
        N = len(texts)
        df: Counter = Counter()
        for text in texts:
            df.update(set(self._get_ngrams(text)))

        # keep top-max_features by document frequency
        top_terms = [t for t, _ in df.most_common(self.max_features)]
        self.vocab_ = {t: i for i, t in enumerate(top_terms)}
        V = len(self.vocab_)

        # smooth IDF
        self.idf_ = np.array(
            [math.log((1 + N) / (1 + df[t])) + 1 for t in top_terms]
        )  # (vocab_size,)
        self._fitted = True
        return self

    def transform(self, texts: List[str]) -> np.ndarray:
        """Return (num_samples, vocab_size) TF-IDF matrix."""
        assert self._fitted
        V = len(self.vocab_)
        X = np.zeros((len(texts), V), dtype=np.float32)
        for i, text in enumerate(texts):
            grams = self._get_ngrams(text)
            tf: Counter = Counter(grams)
            for term, cnt in tf.items():
                if term in self.vocab_:
                    j = self.vocab_[term]
                    X[i, j] = cnt * self.idf_[j]
            # L2 normalise
            norm = np.linalg.norm(X[i])
            if norm > 0:
                X[i] /= norm
        return X  # (num_samples, vocab_size)

    def fit_transform(self, texts: List[str]) -> np.ndarray:
        return self.fit(texts).transform(texts)

In [ ]:
# ── Logistic Regression (SGD, from scratch) ──────────────────────────────
class LogisticRegressionSGD:
    """Binary logistic regression trained with mini-batch SGD.

    Loss  : BCE + (lambda/2)*||w||^2
    Grad w: X^T(y_hat - y)/B + lambda*w
    Grad b: mean(y_hat - y)
    """

    def __init__(
        self,
        lr: float = 0.1,
        l2: float = 1e-3,
        epochs: int = 50,
        batch_size: int = 8,
    ):
        self.lr, self.l2 = lr, l2
        self.epochs, self.batch_size = epochs, batch_size
        self.w: Optional[np.ndarray] = None
        self.b: float = 0.0
        self.losses: List[float] = []

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        """Numerically stable sigmoid: clip to avoid overflow."""
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LogisticRegressionSGD":
        """Train on (num_samples, num_features) X and (num_samples,) binary y."""
        num_samples, num_features = X.shape
        self.w = np.zeros(num_features, dtype=np.float64)
        self.b = 0.0

        idx = np.arange(num_samples)
        for epoch in range(self.epochs):
            np.random.shuffle(idx)
            epoch_loss = 0.0
            batches = 0
            for start in range(0, num_samples, self.batch_size):
                batch_idx = idx[start : start + self.batch_size]
                Xb = X[batch_idx].astype(np.float64)  # (B, F)
                yb = y[batch_idx].astype(np.float64)  # (B,)

                # forward pass
                logits = Xb @ self.w + self.b  # (B,)
                y_hat = self._sigmoid(logits)  # (B,)

                # BCE loss + L2 regularisation
                eps = 1e-9
                bce = -np.mean(
                    yb * np.log(y_hat + eps) + (1 - yb) * np.log(1 - y_hat + eps)
                )
                reg = (self.l2 / 2) * np.dot(self.w, self.w)
                epoch_loss += bce + reg
                batches += 1

                # backward pass
                err = (y_hat - yb) / len(yb)  # (B,)
                dw = Xb.T @ err + self.l2 * self.w  # (F,)
                db = np.sum(err)

                self.w -= self.lr * dw
                self.b -= self.lr * db

            self.losses.append(epoch_loss / max(batches, 1))
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return (num_samples,) probability of positive class."""
        return self._sigmoid(X.astype(np.float64) @ self.w + self.b)

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(X) >= threshold).astype(int)

In [ ]:
# ── train quality classifier ─────────────────────────────────────────────
def train_quality_classifier(
    pos_docs: List[Dict],
    neg_docs: List[Dict],
    vectorizer: TFIDFVectorizer,
    clf: LogisticRegressionSGD,
) -> Tuple[TFIDFVectorizer, LogisticRegressionSGD]:
    """Fit vectorizer + classifier on labelled seed data.

    pos_docs: high-quality examples (label=1)
    neg_docs: low-quality / web-noise examples (label=0)
    """
    texts = [d["text"] for d in pos_docs + neg_docs]
    labels = np.array([1] * len(pos_docs) + [0] * len(neg_docs))

    X = vectorizer.fit_transform(texts)  # (num_samples, vocab_size)
    clf.fit(X, labels)
    return vectorizer, clf


def classifier_filter(
    docs: List[Dict],
    vectorizer: TFIDFVectorizer,
    clf: LogisticRegressionSGD,
    threshold: float = 0.5,
) -> Tuple[List[Dict], List[Dict]]:
    """Score each document and keep those above the quality threshold."""
    texts = [d["text"] for d in docs]
    X = vectorizer.transform(texts)  # (num_samples, vocab_size)
    probs = clf.predict_proba(X)  # (num_samples,)
    for doc, p in zip(docs, probs):
        doc["quality_score"] = float(p)
    passed = [d for d in docs if d["quality_score"] >= threshold]
    rejected = [d for d in docs if d["quality_score"] < threshold]
    return passed, rejected


# seed data from raw corpus (use tier labels as weak supervision)
pos_seed = [d for d in RAW_CORPUS if d["tier"] == "high"]
neg_seed = [d for d in RAW_CORPUS if d["tier"] == "low"]

vec = TFIDFVectorizer(max_features=300)
clf = LogisticRegressionSGD(lr=0.2, l2=1e-3, epochs=80, batch_size=4)
vec, clf = train_quality_classifier(pos_seed, neg_seed, vec, clf)

clf_passed, clf_rejected = classifier_filter(ppl_passed, vec, clf, threshold=0.4)
print(
    f"After classifier filter: {len(clf_passed)} passed, {len(clf_rejected)} rejected"
)

In [ ]:
# ── visualise quality filtering ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

tier_palette = {"high": "#4CAF50", "medium": "#FFC107", "low": "#F44336"}

# perplexity distribution by tier
for tier, color in tier_palette.items():
    ppls = [d["ppl"] for d in deduped_corpus if d["tier"] == tier]
    if ppls:
        axes[0, 0].hist(
            ppls, bins=10, alpha=0.7, color=color, label=tier, edgecolor="white"
        )
axes[0, 0].axvline(PPL_MIN, color="gray", linestyle="--")
axes[0, 0].axvline(PPL_MAX, color="gray", linestyle="--", label="thresholds")
axes[0, 0].set_xscale("log")
axes[0, 0].set_title("Perplexity Distribution by Tier")
axes[0, 0].set_xlabel("Perplexity (log scale)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

# retention rate vs PPL max threshold
thresholds = np.logspace(1, 4, 50)
retention = [
    sum(1 for d in deduped_corpus if PPL_MIN <= d["ppl"] <= t) / len(deduped_corpus)
    for t in thresholds
]
axes[0, 1].plot(thresholds, retention, color="#5c85d6")
axes[0, 1].axvline(PPL_MAX, color="red", linestyle="--", label=f"chosen={PPL_MAX}")
axes[0, 1].set_xscale("log")
axes[0, 1].set_title("Retention Rate vs PPL Max Threshold")
axes[0, 1].set_xlabel("PPL max threshold")
axes[0, 1].set_ylabel("Fraction retained")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# classifier score distribution
for tier, color in tier_palette.items():
    scores = [
        d["quality_score"]
        for d in clf_passed + clf_rejected
        if d.get("quality_score") is not None and d["tier"] == tier
    ]
    if scores:
        axes[1, 0].hist(
            scores, bins=10, alpha=0.7, color=color, label=tier, edgecolor="white"
        )
axes[1, 0].axvline(0.4, color="gray", linestyle="--", label="threshold=0.4")
axes[1, 0].set_title("Classifier Quality Score Distribution")
axes[1, 0].set_xlabel("Quality score")
axes[1, 0].set_ylabel("Count")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# classifier training loss
axes[1, 1].plot(clf.losses, color="#e07b54")
axes[1, 1].set_title("Logistic Regression Training Loss")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("BCE + L2 loss")
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 5) Curriculum Learning and Domain Mixing

## Intuition
**Curriculum learning** (Bengio et al., 2009) presents training examples in a
meaningful order -- typically easy to hard -- mimicking how humans learn.
For LLM pre-training, *difficulty* can be measured along multiple axes:
perplexity of the document relative to a reference LM, vocabulary rarity, or
document length (longer = more context to model).

**Data mixing** controls the proportion of each domain (code, books, web,
Wikipedia) in each training batch.  Temperature-based mixing interpolates
between proportional sampling (T=1) and uniform sampling (T->inf).

    p_i(T) = (n_i/N)^(1/T) / sum_j (n_j/N)^(1/T)

High T -> rare domains get more representation; T=1 -> natural proportions.

**Sample input:** corpus of 15 documents from 3 domains, T=0.7  
**Sample output:** per-domain sampling weights that over-represent rare domains.

In [ ]:
# ── difficulty scorer: multi-axis ────────────────────────────────────────
@dataclass
class DifficultyScorer:
    """Score each document on three axes, then combine into a single score.

    Axes:
      - ppl_zscore     : normalised perplexity (higher PPL = harder)
      - vocab_rarity   : fraction of words not in common-word top-k
      - length_score   : longer documents are harder (z-scored)
    """

    common_vocab_size: int = 500
    ppl_weight: float = 0.5
    rarity_weight: float = 0.3
    length_weight: float = 0.2

    # calibration stats (filled by .calibrate())
    ppl_mean: float = field(default=0.0, init=False)
    ppl_std: float = field(default=1.0, init=False)
    len_mean: float = field(default=0.0, init=False)
    len_std: float = field(default=1.0, init=False)
    common: Set[str] = field(default_factory=set, init=False)

    def calibrate(self, docs: List[Dict]) -> "DifficultyScorer":
        """Compute population stats needed for z-score normalisation."""
        ppls = np.array([d["ppl"] for d in docs])
        lens = np.array([len(d["text"].split()) for d in docs])
        self.ppl_mean, self.ppl_std = ppls.mean(), ppls.std() + 1e-9
        self.len_mean, self.len_std = lens.mean(), lens.std() + 1e-9

        # build common vocabulary from the corpus itself
        all_words: Counter = Counter()
        for d in docs:
            all_words.update(d["text"].lower().split())
        self.common = {w for w, _ in all_words.most_common(self.common_vocab_size)}
        return self

    def score(self, doc: Dict) -> float:
        """Return a [0,1] difficulty score; higher = harder."""
        ppl_z = (doc["ppl"] - self.ppl_mean) / self.ppl_std
        words = doc["text"].lower().split()
        rarity = sum(1 for w in words if w not in self.common) / max(len(words), 1)
        n_words = len(words)
        len_z = (n_words - self.len_mean) / self.len_std

        raw = (
            self.ppl_weight * ppl_z
            + self.rarity_weight * rarity
            + self.length_weight * len_z
        )
        # sigmoid to map to (0, 1)
        return 1.0 / (1.0 + math.exp(-raw))


scorer = DifficultyScorer()
scorer.calibrate(clf_passed)
for doc in clf_passed:
    doc["difficulty"] = scorer.score(doc)

sorted_by_diff = sorted(clf_passed, key=lambda d: d["difficulty"])
print("Documents sorted easy -> hard:")
for d in sorted_by_diff:
    print(
        f"  diff={d['difficulty']:.3f}  tier={d['tier']:6s}  "
        f"ppl={d['ppl']:.1f}  '{d['text'][:60]}...'"
    )

In [ ]:
# ── pacing functions ─────────────────────────────────────────────────────
def linear_pacing(step: int, total_steps: int) -> float:
    """Fraction of corpus unlocked grows linearly with training step."""
    return min(1.0, (step + 1) / total_steps)


def root_pacing(step: int, total_steps: int, power: float = 0.5) -> float:
    """Fraction unlocked grows as a square root -- faster early curriculum."""
    return min(1.0, ((step + 1) / total_steps) ** power)


def geometric_pacing(step: int, total_steps: int, base: float = 2.0) -> float:
    """Fraction unlocked follows geometric (exponential) growth -- very slow start."""
    ratio = (step + 1) / total_steps
    return min(1.0, (base**ratio - 1) / (base - 1))


# ── curriculum sampler ───────────────────────────────────────────────────
class CurriculumSampler:
    """Sort corpus by difficulty and gradually unlock harder documents.

    At each training step, only documents in the unlocked pool are sampled.
    """

    def __init__(self, docs: List[Dict], pacing_fn, total_steps: int):
        self.docs = sorted(docs, key=lambda d: d["difficulty"])  # easy first
        self.pacing_fn = pacing_fn
        self.total_steps = total_steps

    def sample(self, step: int, k: int) -> List[Dict]:
        """Return k documents from the currently unlocked pool."""
        frac = self.pacing_fn(step, self.total_steps)
        pool_size = max(1, int(len(self.docs) * frac))
        pool = self.docs[:pool_size]
        return random.choices(pool, k=min(k, len(pool)))


sampler = CurriculumSampler(clf_passed, root_pacing, total_steps=100)
print("Curriculum sampling demonstration (every 20 steps):")
for step in [0, 20, 40, 60, 80, 99]:
    batch = sampler.sample(step, k=3)
    max_d = max(d["difficulty"] for d in batch)
    frac = root_pacing(step, 100)
    print(f"  step={step:3d}  unlocked={frac:.2f}  max_difficulty={max_d:.3f}")

In [ ]:
# ── temperature-based domain mixing ─────────────────────────────────────
def temperature_mixing(
    domain_sizes: Dict[str, int], temperature: float
) -> Dict[str, float]:
    """Compute per-domain sampling weights with temperature T.

    p_i(T) = (n_i/N)^(1/T) / sum_j (n_j/N)^(1/T)

    T=1  -> proportional to domain size (natural distribution)
    T<1  -> concentrate on larger domains (sharpening)
    T>1  -> spread toward smaller domains (over-sampling rare data)
    """
    N = sum(domain_sizes.values())
    raw = {d: (n / N) ** (1.0 / temperature) for d, n in domain_sizes.items()}
    total = sum(raw.values())
    return {d: v / total for d, v in raw.items()}


# simulate three training domains
DOMAINS = {"wikipedia": 500, "books": 200, "web": 3000}

print(f"{'Temperature':>12} | " + " | ".join(f"{d:>12}" for d in DOMAINS))
print("-" * 60)
for T in [0.5, 0.7, 1.0, 1.5, 2.0, 5.0]:
    weights = temperature_mixing(DOMAINS, T)
    row = " | ".join(f"{weights[d]:>12.3f}" for d in DOMAINS)
    print(f"  T={T:<8.1f} | {row}")

In [ ]:
# ── visualise curriculum and mixing ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

steps = np.arange(100)

# pacing functions
for fn, label, style in [
    (linear_pacing, "linear", "-"),
    (root_pacing, "root", "--"),
    (geometric_pacing, "geometric", ":"),
]:
    fracs = [fn(s, 100) for s in steps]
    axes[0, 0].plot(steps, fracs, style, label=label)
axes[0, 0].set_title("Curriculum Pacing Functions")
axes[0, 0].set_xlabel("Training step")
axes[0, 0].set_ylabel("Fraction of corpus unlocked")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# difficulty trajectory using root pacing
unlocked_diffs = []
for s in steps:
    frac = root_pacing(s, 100)
    pool_end = max(1, int(len(sorted_by_diff) * frac))
    avg_diff = np.mean([d["difficulty"] for d in sorted_by_diff[:pool_end]])
    unlocked_diffs.append(avg_diff)
axes[0, 1].plot(steps, unlocked_diffs, color="#e07b54")
axes[0, 1].set_title("Mean Difficulty of Unlocked Pool (root pacing)")
axes[0, 1].set_xlabel("Training step")
axes[0, 1].set_ylabel("Mean difficulty score")
axes[0, 1].grid(alpha=0.3)

# difficulty distribution of final corpus
diffs = [d["difficulty"] for d in clf_passed]
axes[1, 0].hist(diffs, bins=15, color="#5c85d6", edgecolor="white")
axes[1, 0].set_title("Difficulty Score Distribution")
axes[1, 0].set_xlabel("Difficulty score")
axes[1, 0].set_ylabel("Count")
axes[1, 0].grid(alpha=0.3)

# domain mixing weights at different temperatures
temps = [0.5, 0.7, 1.0, 1.5, 2.0, 5.0]
domain_list = list(DOMAINS.keys())
x = np.arange(len(temps))
width = 0.25
for i, d in enumerate(domain_list):
    weights = [temperature_mixing(DOMAINS, T)[d] for T in temps]
    axes[1, 1].bar(x + i * width, weights, width, label=d, alpha=0.8)
axes[1, 1].set_xticks(x + width)
axes[1, 1].set_xticklabels([str(T) for T in temps])
axes[1, 1].set_title("Domain Mixing Weights vs Temperature")
axes[1, 1].set_xlabel("Temperature")
axes[1, 1].set_ylabel("Sampling weight")
axes[1, 1].legend()
axes[1, 1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# 6) Training Stability: Loss Spikes, Gradient Clipping, LR Warm-up

## Intuition
Large-scale LLM training is notoriously unstable.  Loss spikes (sudden jumps
in training loss) can derail a multi-week training run.  Three mechanisms
work together to prevent them:

1. **Gradient clipping** -- rescale the gradient if its global L2 norm exceeds
   a threshold: `g <- g * (max_norm / ||g||)`.  Prevents a single bad batch
   from causing a catastrophic weight update.

2. **LR warm-up** -- start with a very small learning rate and ramp to the
   target over a few hundred steps.  Prevents large gradients early in training
   when the loss landscape is steep and poorly conditioned.

3. **Cosine decay** -- smoothly reduce LR after warm-up to avoid oscillation
   near the loss minimum.

## Loss spike detection
We use an exponential moving average (EMA) to track the expected loss and its
variance.  A spike is flagged when the current loss deviates more than k
standard deviations above the EMA.

**Sample input:** training run with injected spikes at steps 80, 130, 180  
**Sample output:** spike detection flags + automatic recovery via rollback.

In [ ]:
# ── micro-transformer (from scratch) ─────────────────────────────────────


class MultiHeadAttention(nn.Module):
    """Causal multi-head self-attention with learned projections.

    Follows the GPT-2 architecture: split_heads -> scaled dot-product ->
    causal mask -> softmax -> merge_heads -> output projection.
    """

    def __init__(self, model_dim: int, num_heads: int):
        super().__init__()
        assert model_dim % num_heads == 0, "model_dim must divide num_heads"
        self.model_dim = model_dim
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads

        self.qkv_proj = nn.Linear(model_dim, 3 * model_dim, bias=False)
        self.out_proj = nn.Linear(model_dim, model_dim, bias=False)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, model_dim) -> (batch_num, num_heads, seq_len, head_dim)
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    def merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, num_heads, seq_len, head_dim) -> (batch_num, seq_len, model_dim)
        B, H, T, D = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, H * D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, model_dim) -> (batch_num, seq_len, 3*model_dim)
        qkv = self.qkv_proj(x)

        # (batch_num, seq_len, 3*model_dim) -> three (batch_num, num_heads, seq_len, head_dim)
        q, k, v = qkv.split(self.model_dim, dim=-1)
        q, k, v = self.split_heads(q), self.split_heads(k), self.split_heads(v)

        # scaled dot-product attention
        scale = self.head_dim**-0.5
        # (batch_num, num_heads, seq_len, seq_len)
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale

        # causal mask: upper triangle set to -inf
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn = attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        # (batch_num, num_heads, seq_len, head_dim)
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)

        # (batch_num, num_heads, seq_len, head_dim) -> (batch_num, seq_len, model_dim)
        out = self.merge_heads(out)

        # (batch_num, seq_len, model_dim) -> (batch_num, seq_len, model_dim)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    """Pre-norm transformer block (GPT-2 / LLaMA style).

    Pre-norm (LayerNorm before sublayer) is more stable than post-norm
    because gradients are not compressed through LayerNorm on the way back.
    """

    def __init__(self, model_dim: int, num_heads: int, ffn_mult: int = 4):
        super().__init__()
        hidden_dim = model_dim * ffn_mult
        self.ln1 = nn.LayerNorm(model_dim)
        self.attn = MultiHeadAttention(model_dim, num_heads)
        self.ln2 = nn.LayerNorm(model_dim)
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, model_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, model_dim) -- residual stream preserved throughout
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MicroTransformer(nn.Module):
    """Tiny GPT-style transformer for stability experiments.

    Weight tying: tok_emb.weight == lm_head.weight (reduces parameters,
    standard in GPT-2 and subsequent models).
    """

    def __init__(
        self,
        vocab_size: int,
        model_dim: int = 64,
        num_heads: int = 4,
        num_layers: int = 2,
        max_len: int = 128,
    ):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, model_dim)
        self.pos_emb = nn.Embedding(max_len, model_dim)
        self.blocks = nn.ModuleList(
            [TransformerBlock(model_dim, num_heads) for _ in range(num_layers)]
        )
        self.ln_f = nn.LayerNorm(model_dim)
        self.lm_head = nn.Linear(model_dim, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying

        # GPT-2 initialisation: std=0.02 for most weights
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # idx: (batch_num, seq_len)
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)  # (1, seq_len)

        # (batch_num, seq_len, model_dim)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)

        # (batch_num, seq_len, vocab_size)
        return self.lm_head(x)


print("MicroTransformer defined successfully")

In [ ]:
# ── training utilities: gradient clipping, LR schedule, spike detection ──


def compute_grad_norm(model: nn.Module) -> float:
    """Compute global L2 norm of all parameter gradients.

    Global norm = sqrt( sum_p ||grad_p||^2 )
    This is what torch.nn.utils.clip_grad_norm_ uses internally.
    """
    total_sq = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_sq += p.grad.data.norm(2).item() ** 2
    return math.sqrt(total_sq)


def clip_gradients(model: nn.Module, max_norm: float) -> float:
    """Clip gradients if global norm exceeds max_norm.

    Rescale all gradients by (max_norm / global_norm) when norm > max_norm.
    Returns the pre-clip gradient norm.
    """
    norm = compute_grad_norm(model)
    if norm > max_norm:
        scale = max_norm / (norm + 1e-6)
        for p in model.parameters():
            if p.grad is not None:
                p.grad.data.mul_(scale)
    return norm


class WarmupCosineScheduler:
    """Linear warm-up followed by cosine decay.

    lr(t) = lr_max * t / warmup_steps          for t < warmup_steps
    lr(t) = lr_min + 0.5*(lr_max-lr_min)*(1 + cos(pi*(t-warmup)/(total-warmup)))
                                                for t >= warmup_steps
    """

    def __init__(
        self,
        lr_max: float,
        warmup_steps: int,
        total_steps: int,
        lr_min: float = 0.0,
    ):
        self.lr_max = lr_max
        self.lr_min = lr_min
        self.warmup = warmup_steps
        self.total = total_steps

    def get_lr(self, step: int) -> float:
        if step < self.warmup:
            return self.lr_max * (step + 1) / self.warmup
        progress = (step - self.warmup) / max(1, self.total - self.warmup)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.lr_min + (self.lr_max - self.lr_min) * cosine


class LossSpikeDetector:
    """Online spike detector using exponential moving average + variance.

    EMA of loss: mu_t = alpha*mu_{t-1} + (1-alpha)*loss_t
    EMA of squared loss: v_t tracks E[loss^2]
    Online std: sqrt(v_t - mu_t^2)
    Spike flag: (loss_t - mu_t) > k * std_t
    """

    def __init__(self, alpha: float = 0.95, k: float = 2.5):
        self.alpha = alpha
        self.k = k
        self.mu = None  # EMA of loss
        self.v = None  # EMA of loss^2
        self.step = 0

    def update(self, loss: float) -> bool:
        """Update statistics and return True if a spike is detected."""
        if self.mu is None:
            self.mu, self.v = loss, loss**2
            self.step += 1
            return False
        self.mu = self.alpha * self.mu + (1 - self.alpha) * loss
        self.v = self.alpha * self.v + (1 - self.alpha) * loss**2
        std = math.sqrt(max(0.0, self.v - self.mu**2))
        spike = (loss - self.mu) > self.k * std and std > 0
        self.step += 1
        return spike


print("Training utilities defined successfully")

In [ ]:
# ── character-level dataset ───────────────────────────────────────────────
def build_char_vocab(texts: List[str]) -> Dict[str, int]:
    """Build a character-level vocabulary from a list of texts."""
    chars = sorted(set("".join(texts)))
    return {c: i for i, c in enumerate(chars)}


class CharTextDataset(Dataset):
    """Character-level language modelling dataset.

    Each item is a (seq_len,) input sequence and a (seq_len,) target sequence
    shifted one position to the right.
    """

    def __init__(self, texts: List[str], vocab: Dict[str, int], seq_len: int = 64):
        self.vocab = vocab
        self.seq_len = seq_len
        corpus = " ".join(texts)
        self.data = [vocab.get(c, 0) for c in corpus]

    def __len__(self) -> int:
        return max(0, len(self.data) - self.seq_len - 1)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk = self.data[idx : idx + self.seq_len + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)  # (seq_len,)
        y = torch.tensor(chunk[1:], dtype=torch.long)  # (seq_len,)
        return x, y

In [ ]:
# ── training loop ─────────────────────────────────────────────────────────
def run_training(
    texts: List[str],
    num_steps: int = 200,
    lr_max: float = 3e-3,
    warmup_steps: int = 20,
    max_grad_norm: float = 1.0,
    use_clip: bool = True,
    use_warmup: bool = True,
    inject_spike_steps: Optional[List[int]] = None,
    recover_on_spike: bool = True,
    label: str = "run",
) -> Dict:
    """Full training loop for the MicroTransformer.

    Logs per-step loss, gradient norm, and detected spike flags.
    inject_spike_steps: list of steps at which to artificially multiply
        the gradient by 50 times, simulating a bad batch.
    """
    vocab = build_char_vocab(texts)
    dataset = CharTextDataset(texts, vocab, seq_len=64)
    loader = DataLoader(dataset, batch_size=4, shuffle=True, drop_last=True)

    model = MicroTransformer(
        vocab_size=len(vocab), model_dim=64, num_heads=4, num_layers=2
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr_max, betas=(0.9, 0.95), weight_decay=0.1
    )
    scheduler = WarmupCosineScheduler(lr_max, warmup_steps, num_steps)
    detector = LossSpikeDetector(alpha=0.95, k=2.5)

    inject_steps = set(inject_spike_steps or [])
    history = {"loss": [], "grad_norm": [], "lr": [], "spike": []}

    best_state = {k: v.clone() for k, v in model.state_dict().items()}
    best_loss = float("inf")
    data_iter = iter(loader)
    step = 0

    while step < num_steps:
        try:
            xb, yb = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            xb, yb = next(data_iter)

        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        model.train()

        # set learning rate for this step
        current_lr = scheduler.get_lr(step) if use_warmup else lr_max
        for pg in optimizer.param_groups:
            pg["lr"] = current_lr

        optimizer.zero_grad()

        # (batch_num, seq_len, vocab_size)
        logits = model(xb)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1))
        loss.backward()

        # artificial spike injection for demonstration
        if step in inject_steps:
            for p in model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(50.0)

        grad_norm = compute_grad_norm(model)
        if use_clip:
            clip_gradients(model, max_grad_norm)

        optimizer.step()

        loss_val = loss.item()
        is_spike = detector.update(loss_val)

        if is_spike and recover_on_spike:
            model.load_state_dict(best_state)  # rollback to last good state
        elif loss_val < best_loss:
            best_loss = loss_val
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        history["loss"].append(loss_val)
        history["grad_norm"].append(grad_norm)
        history["lr"].append(current_lr)
        history["spike"].append(is_spike)
        step += 1

    return history


# gather training texts from the curated corpus
train_texts = [d["text"] for d in clf_passed] * 5  # repeat to have enough data
if not train_texts:
    # fallback if all docs were filtered out
    train_texts = [d["text"] for d in RAW_CORPUS if d["tier"] == "high"] * 10

SPIKE_STEPS = [80, 130, 180]
print("Running STABLE training  (clip=True, warmup=True, recovery=True) ...")
hist_stable = run_training(
    train_texts,
    num_steps=200,
    lr_max=3e-3,
    warmup_steps=20,
    max_grad_norm=1.0,
    use_clip=True,
    use_warmup=True,
    inject_spike_steps=SPIKE_STEPS,
    recover_on_spike=True,
    label="stable",
)

print("Running UNSTABLE training (clip=False, warmup=False, recovery=False) ...")
hist_unstable = run_training(
    train_texts,
    num_steps=200,
    lr_max=3e-3,
    warmup_steps=20,
    max_grad_norm=1.0,
    use_clip=False,
    use_warmup=False,
    inject_spike_steps=SPIKE_STEPS,
    recover_on_spike=False,
    label="unstable",
)
print("Training complete!")

In [ ]:
# ── visualise training stability ──────────────────────────────────────────
fig = plt.figure(figsize=(15, 12))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

steps = range(200)


def smooth(vals, w=10):
    """Simple moving average for visualisation."""
    out = []
    for i in range(len(vals)):
        start = max(0, i - w + 1)
        out.append(np.mean(vals[start : i + 1]))
    return out


# raw loss comparison
ax = fig.add_subplot(gs[0, :])
ax.plot(
    steps,
    hist_stable["loss"],
    color="#4CAF50",
    alpha=0.5,
    linewidth=0.8,
    label="stable",
)
ax.plot(
    steps,
    hist_unstable["loss"],
    color="#F44336",
    alpha=0.5,
    linewidth=0.8,
    label="unstable",
)
for s in SPIKE_STEPS:
    ax.axvline(s, color="gray", linestyle="--", alpha=0.4)
# mark spike detection
for i, is_spike in enumerate(hist_stable["spike"]):
    if is_spike:
        ax.axvline(i, color="#4CAF50", linestyle=":", alpha=0.8)
ax.set_title(
    "Training Loss: Stable vs Unstable (dashed = injected spike, dotted = detected spike)"
)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
ax.set_ylim(0, min(15, max(hist_unstable["loss"]) * 1.1))
ax.grid(alpha=0.3)

# LR schedule
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(steps, hist_stable["lr"], color="#5c85d6")
ax2.set_title("LR Schedule (Warmup + Cosine Decay)")
ax2.set_xlabel("Step")
ax2.set_ylabel("Learning rate")
ax2.grid(alpha=0.3)

# gradient norms
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(
    steps,
    hist_stable["grad_norm"],
    color="#4CAF50",
    alpha=0.7,
    label="stable (clipped)",
)
ax3.plot(
    steps, hist_unstable["grad_norm"], color="#F44336", alpha=0.7, label="unstable"
)
ax3.axhline(1.0, color="gray", linestyle="--", label="clip_norm=1.0")
for s in SPIKE_STEPS:
    ax3.axvline(s, color="gray", linestyle="--", alpha=0.4)
ax3.set_title("Gradient Norm per Step")
ax3.set_xlabel("Step")
ax3.set_ylabel("Grad norm")
ax3.legend(fontsize=8)
ax3.set_ylim(0, min(25, max(hist_unstable["grad_norm"]) * 1.1))
ax3.grid(alpha=0.3)

# warmup ablation
ax4 = fig.add_subplot(gs[2, 0])
for warmup in [0, 5, 20, 50]:
    sched = WarmupCosineScheduler(3e-3, warmup, 200)
    lrs = [sched.get_lr(s) for s in range(200)]
    ax4.plot(range(200), lrs, label=f"warmup={warmup}")
ax4.set_title("Effect of Warmup Length on LR Schedule")
ax4.set_xlabel("Step")
ax4.set_ylabel("Learning rate")
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

# smoothed loss
ax5 = fig.add_subplot(gs[2, 1])
ax5.plot(steps, smooth(hist_stable["loss"]), color="#4CAF50", label="stable (smoothed)")
ax5.plot(
    steps, smooth(hist_unstable["loss"]), color="#F44336", label="unstable (smoothed)"
)
ax5.set_title("Smoothed Loss (window=10)")
ax5.set_xlabel("Step")
ax5.set_ylabel("Loss")
ax5.legend(fontsize=8)
ax5.grid(alpha=0.3)

plt.suptitle("Training Stability Analysis", fontsize=14, y=1.01)
plt.show()

# summary table
print("\n=== Summary ===")
print(f"{'Metric':<30} {'Stable':>12} {'Unstable':>12}")
print("-" * 55)
print(
    f"{'Final loss (last 20 avg)':<30} "
    f"{np.mean(hist_stable['loss'][-20:]):>12.3f} "
    f"{np.mean(hist_unstable['loss'][-20:]):>12.3f}"
)
print(
    f"{'Max gradient norm':<30} "
    f"{max(hist_stable['grad_norm']):>12.2f} "
    f"{max(hist_unstable['grad_norm']):>12.2f}"
)
print(
    f"{'Spikes detected':<30} "
    f"{sum(hist_stable['spike']):>12} "
    f"{sum(hist_unstable['spike']):>12}"
)

# 7) End-to-End Pipeline

Running all five stages in sequence on the raw corpus and printing the data
funnel at each stage.

In [ ]:
# ── end-to-end pipeline function ─────────────────────────────────────────
def run_complete_pipeline(
    raw_corpus: List[Dict],
    filter_cfg: FilterConfig,
    lsh_cfg: LSHConfig,
    ppl_min: float = 4.0,
    ppl_max: float = 600.0,
    clf_threshold: float = 0.4,
    temperature: float = 0.7,
) -> Dict:
    """Execute all five pipeline stages and return the curated corpus + stats."""
    counts = {"raw": len(raw_corpus)}
    print(f"Stage 0 | Raw                 : {counts['raw']:>4} documents")

    # stage 1: heuristic filters
    stage1 = [d for d in raw_corpus if apply_heuristic_filters(d, filter_cfg)[0]]
    counts["heuristic"] = len(stage1)
    print(f"Stage 1 | Heuristic filters   : {counts['heuristic']:>4} documents")

    # stage 2a: exact dedup
    stage2a = exact_dedup(stage1)
    counts["exact_dedup"] = len(stage2a)
    print(f"Stage 2a| Exact dedup         : {counts['exact_dedup']:>4} documents")

    # stage 2b: MinHash LSH fuzzy dedup
    if len(stage2a) >= 2:
        sig_m, d_ids = build_minhash_signatures(stage2a, lsh_cfg)
        buckets2 = lsh_band_buckets(sig_m, lsh_cfg)
        cands2 = find_candidate_pairs(buckets2)
        stage2b = fuzzy_dedup(stage2a, sig_m, d_ids, cands2, lsh_cfg)
    else:
        stage2b = stage2a
    counts["fuzzy_dedup"] = len(stage2b)
    print(f"Stage 2b| Fuzzy dedup         : {counts['fuzzy_dedup']:>4} documents")

    # stage 3: perplexity filter (reuse existing LM)
    for d in stage2b:
        d["ppl"] = lm.perplexity(d["text"])
    stage3, _ = perplexity_filter(stage2b, ppl_min, ppl_max)
    counts["ppl_filter"] = len(stage3)
    print(f"Stage 3 | Perplexity filter   : {counts['ppl_filter']:>4} documents")

    # stage 4: classifier filter (reuse existing vectorizer + clf)
    stage4, _ = classifier_filter(stage3, vec, clf, threshold=clf_threshold)
    counts["clf_filter"] = len(stage4)
    print(f"Stage 4 | Classifier filter   : {counts['clf_filter']:>4} documents")

    # stage 5: domain mixing weights
    domain_counts: Counter = Counter(d.get("tier", "unknown") for d in stage4)
    mix_weights = temperature_mixing(
        {k: v for k, v in domain_counts.items()}, temperature
    )
    print(f"Stage 5 | Domain mixing (T={temperature}) : {mix_weights}")

    return {"corpus": stage4, "counts": counts, "mixing": mix_weights}


result = run_complete_pipeline(RAW_CORPUS, CFG, LSH_CFG)

In [ ]:
# ── funnel visualisation ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

stages = list(result["counts"].keys())
counts = list(result["counts"].values())
colors = ["#5c85d6", "#4CAF50", "#FFC107", "#e07b54", "#9C27B0", "#00BCD4"][
    : len(stages)
]

bars = ax.barh(
    [s.replace("_", " ").title() for s in stages[::-1]],
    counts[::-1],
    color=colors[::-1],
    edgecolor="white",
)
for bar, val in zip(bars, counts[::-1]):
    ax.text(
        val + 0.1,
        bar.get_y() + bar.get_height() / 2,
        str(val),
        va="center",
        fontsize=10,
    )

ax.set_xlabel("Documents remaining")
ax.set_title("Full Data Curation Pipeline Funnel")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal curated corpus: {len(result['corpus'])} documents")
print(
    f"Total reduction: {len(RAW_CORPUS)} -> {len(result['corpus'])} "
    f"({100*len(result['corpus'])/len(RAW_CORPUS):.1f}% retained)"
)

# 8) Key Takeaways and Further Reading

## Summary Table

| Technique | Algorithm | Complexity | Key Paper |
|---|---|---|---|
| Heuristic filters | Word count, symbol ratio, n-gram dup | O(n*L) | Gopher (Rae et al. 2021) |
| Exact deduplication | SHA-256 hash set | O(n) | Lee et al. 2022 |
| MinHash signatures | Universal hashing, min over shingles | O(n*H*S) | Broder 1997 |
| LSH banding | Band hash buckets | O(n*H) | Indyk & Motwani 1998 |
| Union-Find clustering | Path compression + rank | O(n*alpha(n)) | Tarjan 1975 |
| Perplexity filtering | n-gram LM, add-k smoothing | O(n*L) | CCNet (Wenzek et al. 2020) |
| Classifier filtering | TF-IDF + Logistic Regression | O(n*V) | Gopher, LLaMA |
| Curriculum learning | Difficulty scoring + pacing | O(n log n) | Bengio et al. 2009 |
| Temperature mixing | Softmax over domain counts | O(D) | T5 (Raffel et al. 2020) |
| Gradient clipping | Global L2 norm rescaling | O(P) | Pascanu et al. 2013 |
| LR warm-up + cosine | Linear ramp + cosine anneal | O(1) | BERT, GPT-2 |
| Loss spike detection | EMA + online variance | O(1) | Biderman et al. 2023 |

Where: n=documents, L=document length, H=num_hashes, S=shingle count,
       V=vocab size, D=num domains, P=num parameters.

## Further Reading
- **CCNet** (Wenzek et al., 2020): language ID + perplexity filtering at scale
- **Gopher** (Rae et al., 2021): heuristic filters + quality evaluation
- **LLaMA** (Touvron et al., 2023): fastText classifier-based quality filtering
- **Dolma** (Soldaini et al., 2024): open data pipeline at 3T tokens
- **RedPajama-v2** (Together AI, 2023): 20+ quality signals per document
- **SemDeDup** (Abbas et al., 2023): semantic (embedding-space) deduplication
- **DataComp-LM** (Li et al., 2024): controlled experiments on data curation